In [2]:
import pandas as pd
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

DATA = Path("../data")

df = pd.read_csv(DATA / 'ml_sessions.csv')

In [4]:
df['connector_type'].value_counts()

connector_type
Type2 AC         14833
CCS2             13110
Bharat AC-001     6711
CHAdeMO           5180
Bharat DC-001     2671
GB/T              2495
Name: count, dtype: int64

In [8]:
numeric = [
    'station_age_years', 'power_kw', 'num_bays', 'hour',
    'day_of_week', 'is_weekend', 'ambient_temp_c', 'grid_load_index',
    'start_soc_pct', 'battery_capacity_kwh'
]

categorical = ['connector_type', 'zone', 'vehicle_segment']

X, y = df[numeric + categorical], df['failed']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, random_state=42, test_size=0.25, stratify=y
)

print(X_train[categorical].nunique())
print(X_train[categorical].head(3))

connector_type     6
zone               5
vehicle_segment    4
dtype: int64
      connector_type        zone vehicle_segment
43101           CCS2  Kukatpally              2W
28670           CCS2  Gachibowli              3W
21772           CCS2  Kukatpally              4W


In [9]:
try:
    LogisticRegression().fit(X_train, y_train)

except Exception as e:
    print(type(e).__name__ + ":", e)

ValueError: could not convert string to float: 'CCS2'


In [12]:
from sklearn.preprocessing import OrdinalEncoder

ordinal = OrdinalEncoder().fit(X_train[['connector_type']])

print("codes 0, 1, 2, and so on :", list(ordinal.categories_))

rate = y_train.groupby(X_train['connector_type']).mean()
print(rate.round(4))

codes 0, 1, 2, and so on : [array(['Bharat AC-001', 'Bharat DC-001', 'CCS2', 'CHAdeMO', 'GB/T',
       'Type2 AC'], dtype=object)]
connector_type
Bharat AC-001    0.0697
Bharat DC-001    0.1107
CCS2             0.0813
CHAdeMO          0.1565
GB/T             0.1555
Type2 AC         0.0523
Name: failed, dtype: float64


In [13]:
df['connector_type'].value_counts()

connector_type
Type2 AC         14833
CCS2             13110
Bharat AC-001     6711
CHAdeMO           5180
Bharat DC-001     2671
GB/T              2495
Name: count, dtype: int64

In [16]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(sparse_output=False, handle_unknown = 'ignore')
ohe.fit(X_train[['connector_type']])

print(ohe.get_feature_names_out())

print(X_train['connector_type'].iloc[100], "->", ohe.transform(X_train[['connector_type']].iloc[[100]]))

['connector_type_Bharat AC-001' 'connector_type_Bharat DC-001'
 'connector_type_CCS2' 'connector_type_CHAdeMO' 'connector_type_GB/T'
 'connector_type_Type2 AC']
Type2 AC -> [[0. 0. 0. 0. 0. 1.]]


## Category nobody has seen

In [17]:
new_session = pd.DataFrame({'connector_type': ['ABCD']})
print(ohe.transform(new_session))

[[0. 0. 0. 0. 0. 0.]]


In [19]:
df['vehicle_segment'].value_counts()

vehicle_segment
2W     18840
4W     15410
3W      8002
Bus     2748
Name: count, dtype: int64

In [20]:
size_order = OrdinalEncoder(categories=[['2W', '3W', '4W', 'Bus']])

size_order.fit(X_train[['vehicle_segment']])
print(size_order.transform(pd.DataFrame({'vehicle_segment': ['2W', '4W', 'Bus']})))

[[0.]
 [2.]
 [3.]]


## Column transformer

In [24]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

ct = ColumnTransformer([
    ("num", StandardScaler(), numeric),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical),
])

ct.fit(X_train)

print("columns in :", X_train.shape[1])
print("columns out: ", ct.transform(X_train).shape[1])


columns in : 13
columns out:  25


In [26]:
X_train

,station_age_years,power_kw,num_bays,hour,day_of_week,is_weekend,ambient_temp_c,grid_load_index,start_soc_pct,battery_capacity_kwh,connector_type,zone,vehicle_segment
43101,5.0,60.0,3,9,2,0,25.4,0.643,26.5,2.6,CCS2,Kukatpally,2W
28670,2.0,60.0,2,15,1,0,38.9,0.514,30.9,9.3,CCS2,Gachibowli,3W
21772,5.0,60.0,3,10,6,1,33.2,0.616,48.5,33.6,CCS2,Kukatpally,4W
13036,5.0,60.0,3,17,0,0,25.4,0.701,47.4,3.8,CCS2,Kukatpally,2W
5610,2.0,15.0,1,6,5,1,24.1,0.463,48.9,41.1,Type2 AC,Kukatpally,4W
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6445,7.0,120.0,7,19,3,0,27.3,0.871,65.0,11.5,CCS2,Madhapur,3W
15003,2.0,15.0,1,22,4,0,25.3,0.719,68.6,42.8,Type2 AC,Kukatpally,4W
42177,2.0,15.0,1,21,0,0,26.4,0.694,66.2,62.5,Type2 AC,Kukatpally,4W
24508,4.0,30.0,4,2,0,0,14.8,0.464,41.3,48.6,Bharat AC-001,Gachibowli,4W


In [22]:
for cat in categorical:
    print(df[cat].value_counts())

connector_type
Type2 AC         14833
CCS2             13110
Bharat AC-001     6711
CHAdeMO           5180
Bharat DC-001     2671
GB/T              2495
Name: count, dtype: int64
zone
Gachibowli      10262
Madhapur        10141
Kukatpally       9687
Uppal            7462
Secunderabad     7448
Name: count, dtype: int64
vehicle_segment
2W     18840
4W     15410
3W      8002
Bus     2748
Name: count, dtype: int64


In [23]:
X_train.shape

(33750, 13)

In [35]:
ct.transform(X_train)

array([[ 0.29873011, -0.29489121, -0.26513633, ...,  0.        ,
         0.        ,  0.        ],
       [-1.45306089, -0.29489121, -0.79284472, ...,  1.        ,
         0.        ,  0.        ],
       [ 0.29873011, -0.29489121, -0.26513633, ...,  0.        ,
         1.        ,  0.        ],
       ...,
       [-1.45306089, -1.06493648, -1.32055311, ...,  0.        ,
         1.        ,  0.        ],
       [-0.28520023, -0.80825472,  0.26257206, ...,  0.        ,
         1.        ,  0.        ],
       [-0.86913056,  1.24519932,  0.79028045, ...,  0.        ,
         1.        ,  0.        ]], shape=(33750, 25))

In [34]:
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)

ct.set_output(transform = 'default')
seen = ct.transform(X_train)
#print(seen.iloc[:3, [0, 7, 10, 12]].round(3))


In [30]:
X_train.head(3)

,station_age_years,power_kw,num_bays,hour,day_of_week,is_weekend,ambient_temp_c,grid_load_index,start_soc_pct,battery_capacity_kwh,connector_type,zone,vehicle_segment
43101,5.0,60.0,3,9,2,0,25.4,0.643,26.5,2.6,CCS2,Kukatpally,2W
28670,2.0,60.0,2,15,1,0,38.9,0.514,30.9,9.3,CCS2,Gachibowli,3W
21772,5.0,60.0,3,10,6,1,33.2,0.616,48.5,33.6,CCS2,Kukatpally,4W


In [37]:
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train, test_size=0.25, random_state=42, stratify=y_train
)

numbers_only = ColumnTransformer([('num', StandardScaler(), numeric)])
with_ordinal = ColumnTransformer([
    ('num', StandardScaler(), numeric),
    ('cat', OrdinalEncoder(), categorical)
])

use_onehot = ColumnTransformer([
    ('num', StandardScaler(), numeric),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical)
])

In [38]:
def validation_auc(prep):
    model = LogisticRegression(max_iter=1000)
    model.fit(prep.fit_transform(X_fit), y_fit)
    return roc_auc_score(y_val, model.predict_proba(prep.transform(X_val))[:, 1])

auc_num = validation_auc(numbers_only)
auc_ord = validation_auc(with_ordinal)
auc_ohe = validation_auc(use_onehot)

print(round(auc_num, 4))
print(round(auc_ord, 4))
print(round(auc_ohe, 4))


0.6848
0.6843
0.6926


## Exercises to work on

1. use drop = 'if_binary' in OHE and check which columns change and why none over here ?

2. Use OrdinalEncoder with the real size 'vehicle_segment' inside the column transformer and check the AUC whether it is moving up or down ?

3. print ct.transformers_ after fitting, and check what you see for each group ?
